# 03 - ניתוח תיאורי של הרשת

מחברת זו לוקחת את גרף סמיכות-הנסיעות (trip-adjacency graph) שנבנה במחברת `02_graph_construction` ומשיבה על השאלה הראשונה שכל מחקר רשתות חייב לענות עליה בטרם יוכל לדון בחוסן: **כיצד נראית הרשת הזו בפועל?** אנו מודדים גודל (צמתים, קשתות), דלילות (density, דרגה ממוצעת), פיצול (רכיבי קשירות, חלקו של הרכיב הגדול ביותר), צורת התפלגות הדרגות (היסטוגרמה לינארית לצד תצוגת log-log לבדיקת קיומו של זנב כבד), ואת שני מבני הקשירות המקומית הקלאסיים - **נקודות חיתוך (articulation points)**, כלומר צמתים שהסרתם מנתקת את הגרף, ו-**גשרים (bridges)**, כלומר קשתות שהסרתן מנתקת את הגרף. שני האחרונים הם החוליה המקשרת הישירה לשאלת המחקר של הפרויקט, משום שהם המקומות שבהם לרשת אין כל יתירות.

**שאלת המחקר הנדונה כאן:** עד כמה דלילה ועד כמה שברירית רשת התחבורה הציבורית בישראל ברמה המבנית, וכמה תחנות בודדות / מקטעים בודדים קיימים שאובדנם יפצל את הרשת?

## קלט

* `outputs/nb/02_graph_construction/nodes.csv` - שורה אחת לכל תחנה פעילה, עם `stop_id, stop_name, lat, lon, region, metro`.
* `outputs/nb/02_graph_construction/edges.csv` - שורה אחת לכל מקטע **מכוון** `from_stop, to_stop, trip_frequency`.

שני הקבצים מופקים על ידי מחברת `02_graph_construction`. מחברת זו **אינה** קוראת את קובץ ה-GTFS הגולמי ו**אינה** זקוקה לקובץ `stop_times.txt` בנפח 816 MB.

## פלט

כל הפלטים נכתבים תחת `outputs/nb/03_descriptive_analysis/`:

* `network_summary.json` - כל הסטטיסטיקות המרכזיות במילון אחד.
* `tables/network_summary.csv` - אותן סטטיסטיקות כטבלה בת שורה אחת.
* `tables/degree_distribution.csv` - מספר התחנות לכל ערך דרגה.
* `tables/component_sizes.csv` - גודלו של כל רכיב קשירות.
* `tables/articulation_points.csv` - כל צומת חיתוך, בצירוף שם / קואורדינטות / מחוז / דרגה.
* `tables/bridges.csv` - כל קשת חיתוך, בצירוף שמות הקצוות ותדירות הנסיעות.
* `tables/stops_by_region.csv` - מספר התחנות לכל מחוז מנהלי.
* `figures/degree_distribution.png`, `figures/components_summary_bar.png`, `figures/top_articulation_points.png`, `figures/network_overview_map.png`, `figures/stops_by_region.png`.

דבר מחוץ ל-`outputs/nb/03_descriptive_analysis/` אינו משתנה.

## 1. אתחול סביבת העבודה

התא שלהלן מאפשר להריץ את המחברת הן על עותק מקומי של המאגר והן ב-Google Colab. הוא מגדיר את `_ensure(...)`, המתקין באמצעות pip רק את החבילות שחסרות באמת (כך שהרצה חוזרת של המחברת זולה), ואת `find_repo_root()`, המטפסת מהתיקייה הנוכחית כלפי מעלה בחיפוש אחר תיקיית ה-GTFS, ואם לא נמצאה - משכפלת את המאגר אל `/content`. לאחר מכן הוא מגדיר את `REPO`, `DATA` ו-`OUT` ויוצר את תיקיית הפלט הראשית של המחברת. כל התאים שבהמשך מסתמכים על שלושת הנתיבים הללו, ולכן תא זה חייב לרוץ ראשון.

In [ ]:
# --- Environment bootstrap (safe to re-run, works locally and on Google Colab) ---
import os, sys, subprocess
from pathlib import Path

def _ensure(*pkgs):
    """Install only the packages that are actually missing."""
    import importlib.util
    alias = {"scikit-learn": "sklearn", "python-louvain": "community",
             "python-bidi": "bidi", "node2vec": "node2vec"}
    missing = [p for p in pkgs
               if importlib.util.find_spec(alias.get(p, p.replace("-", "_"))) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

def find_repo_root():
    """Find the repo locally; on Colab, clone it."""
    here = Path(os.getcwd()).resolve()
    for cand in [here, *here.parents]:
        if (cand / "israel-public-transportation").is_dir():
            return cand
    target = Path("/content/israel-transit-network-resilience")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/seanfourman/israel-transit-network-resilience.git",
                        str(target)], check=True)
    return target

REPO = find_repo_root()
os.chdir(REPO)
DATA = REPO / "israel-public-transportation"
OUT = REPO / "outputs" / "nb"
OUT.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO)

## 2. ספריות, תיקיות השלב וקבועים הניתנים לכוונון

אנו מתקינים ומייבאים את מחסנית הכלים המדעית (`pandas`, `numpy`, `networkx`, `matplotlib`, `seaborn`) ולאחר מכן מקבעים את מבנה התיקיות עבור שלב זה. בהתאם למוסכמת הפרויקט, כל מחברת מחזיקה בדיוק בתיקיית פלט אחת: מחברת זו כותבת אל `outputs/nb/03_descriptive_analysis/` (עם תת-התיקיות `tables/` ו-`figures/`) וקוראת את השלב הקודם מתוך `outputs/nb/02_graph_construction/`. התיקיות הקיימות `outputs/tables`, `outputs/figures` ו-`outputs/rail` מחזיקות את התוצאות המצוטטות בדוח הכתוב, ובמכוון לעולם אין כותבים אליהן כאן.

הקבועים מרוכזים כאן כדי שהבודק יוכל להחליף זמן ריצה ברמת פירוט במקום אחד. אף אחד מהאלגוריתמים במחברת זו אינו יקר חישובית - רכיבי קשירות, נקודות חיתוך וגשרים הם כולם שגרות DFS בזמן לינארי ומסתיימות בתוך שניות ספורות על גרף בן 30k צמתים / 52k קשתות - ולכן הקבועים שולטים בעיקר בגודל האיורים ובמספר השורות המוצגות בטבלאות ה-top-N. `FIG_DPI` הוא כפתור העלות האמיתי היחיד: ב-150 dpi, רסטור המפה בת 30,000 הנקודות אורך מספר שניות.

In [ ]:
# --- Libraries and stage folders ------------------------------------------
_ensure('pandas', 'numpy', 'networkx', 'matplotlib', 'seaborn')

import json
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', font_scale=1.05)

PREV = OUT / '02_graph_construction'      # read-only: artifacts of notebook 02
STAGE = OUT / '03_descriptive_analysis'   # everything this notebook produces
TABLES = STAGE / 'tables'
FIGURES = STAGE / 'figures'
TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

# --- Tunable constants ----------------------------------------------------
FIG_DPI = 150        # figure resolution; drop to 90 for faster, smaller files
TOP_N = 20           # rows shown in every top-N table / bar chart
MAP_ALPHA = 0.55     # point transparency on the ~30k-point geographic scatter
DEG_HIST_BINS = 60   # bins of the linear degree histogram

print('previous stage :', PREV)
print('this stage     :', STAGE)

## 3. עיבוד תוויות בעברית

שמות התחנות ב-GTFS הישראלי הם בעברית, ושניים מהאיורים שלהלן (תרשים נקודות החיתוך המובילות והתרשים לפי מחוז) מדפיסים אותם. matplotlib אינה מממשת את אלגוריתם הדו-כיווניות (bidirectional) של Unicode, ולכן טקסט מימין-לשמאל מוצג הפוך ובלתי קריא. התא שלהלן מבצע monkey-patch חד-פעמי ל-`matplotlib.text.Text.set_text`, כך שכל מחרוזת המכילה תווים עבריים מומרת לסדר תצוגה באמצעות `python-bidi` לפני ציורה, ובוחר גופן שאכן מכיל גליפים עבריים (Arial ב-Windows, DejaVu Sans בכל מערכת אחרת). התא הוא אידמפוטנטי - הרצה חוזרת שלו לא תערים patch על patch. כל שאר הטקסט במחברת הוא באנגלית, בהתאם לדרישות ההגשה.

In [ ]:
# Stop names are Hebrew. Matplotlib does not apply the Unicode bidi algorithm, so
# Hebrew labels render reversed. Patch it once, before drawing any figure.
_ensure("python-bidi")
import re
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.text as mtext
from bidi.algorithm import get_display

_HEBREW_RE = re.compile(r"[\u0590-\u05FF]")

def fix_he(text):
    """Return display-ordered text. Non-Hebrew is returned untouched."""
    if not isinstance(text, str) or not _HEBREW_RE.search(text):
        return text
    return get_display(text)

def install_hebrew():
    # Arial exists on Windows; DejaVu Sans ships with matplotlib and covers Hebrew.
    matplotlib.rcParams["font.family"] = ["Arial", "DejaVu Sans"]
    matplotlib.rcParams["axes.unicode_minus"] = False
    if getattr(mtext.Text, "_bidi_patched", False):
        return
    _orig = mtext.Text.set_text
    def set_text(self, s):
        if isinstance(s, str) and getattr(self, "_bidi_display", None) == s:
            return _orig(self, s)
        fixed = fix_he(s)
        if isinstance(fixed, str):
            self._bidi_display = fixed
        return _orig(self, fixed)
    mtext.Text.set_text = set_text
    mtext.Text._bidi_patched = True

install_hebrew()

## 4. טעינת הגרף שהופק במחברת 02

שלב זה תלוי במחברת `02_graph_construction`. במקום לבצע un-pickle לאובייקט `networkx` (קובצי pickle רגישים לגרסה ואינם קריאים לבודק), אנו טוענים מחדש את שתי טבלאות ה-CSV הפשוטות ששלב 02 מייצא ובונים מהן מחדש את אובייקטי הגרף - קובצי ה-CSV מכילים בדיוק את אותו המידע. `find_artifact` סורקת את כל תיקיית השלב הקודם, כך שהיא עובדת בין אם שלב 02 הניח את הטבלאות בשורש התיקייה ובין אם בתוך `tables/`, ומעלה שגיאה מפורשת וברת-פעולה אם התוצרים חסרים.

In [ ]:
# --- Locate the artifacts written by notebook 02 --------------------------
def find_artifact(stage_dir, filename):
    """Return the path of `filename` under a stage folder, or None if absent."""
    if not stage_dir.is_dir():
        return None
    direct = stage_dir / filename
    if direct.exists():
        return direct
    matches = sorted(stage_dir.rglob(filename))
    return matches[0] if matches else None

nodes_path = find_artifact(PREV, 'nodes.csv')
edges_path = find_artifact(PREV, 'edges.csv')
missing = [name for name, p in [('nodes.csv', nodes_path), ('edges.csv', edges_path)] if p is None]
if missing:
    raise FileNotFoundError(
        f"{', '.join(missing)} not found under {PREV} - "
        'run notebook 02_graph_construction first; it writes nodes.csv and edges.csv.'
    )

nodes_df = pd.read_csv(nodes_path, dtype={'stop_id': str}, encoding='utf-8-sig')
edges_df = pd.read_csv(edges_path, dtype={'from_stop': str, 'to_stop': str}, encoding='utf-8-sig')
print(f'nodes: {len(nodes_df):,} rows  <-  {nodes_path}')
print(f'edges: {len(edges_df):,} rows  <-  {edges_path}')
nodes_df.head()

## 5. בנייה מחדש של הגרף המכוון והלא-מכוון

המודל של הפרויקט הוא **גרף סמיכות-נסיעות (trip-adjacency graph)**: צומת הוא תחנה המופיעה בלפחות נסיעה אחת, וקשת מכוונת `u -> v` קיימת כאשר איזושהי נסיעה פוקדת את `v` מיד לאחר `u`; משקל הקשת הוא מספר הנסיעות המשתמשות במקטע זה. `edges.csv` מאחסן את הגרף המכוון הזה. ממנו אנו גוזרים את ההיטל הלא-מכוון `G`, המשמש לכל עבודת הקשירות: קשת אחת לכל זוג לא-מסודר, כאשר משקלי שני כיווני הנסיעה מסוכמים - זו הסיבה שמספר הקשתות הלא-מכוונות (כ-51.8k) קטן מעט ממספר הקשתות המכוונות (כ-52.0k), שכן רוב המקטעים מופעלים בשני הכיוונים ומתמזגים לקשת אחת.

תכונות התחנות (שם, קואורדינטות, מחוז, מטרופולין) מצורפות מתוך `nodes.csv`. הפונקציות העוזרות `_num` ו-`_txt` ממירות ערכים ריקים ו-`NaN` בבטחה, כך שקואורדינטה חסרה הופכת ל-`None` ולא ל-`NaN` שקט שהיה מצויר בהמשך במיקום חסר משמעות.

In [ ]:
# --- Rebuild the two graph objects ----------------------------------------
def _num(value):
    """Coerce to float; return None for blanks, NaN or non-numeric input."""
    if value is None:
        return None
    try:
        f = float(value)
    except (TypeError, ValueError):
        return None
    return None if not np.isfinite(f) else f


def _txt(value):
    """Coerce to a plain string; NaN and None become an empty string."""
    if value is None or (isinstance(value, float) and not np.isfinite(value)):
        return ''
    return str(value)


def build_graphs(nodes_df, edges_df):
    """Return (G, D): the undirected projection and the directed trip graph."""
    attr = {}
    for rec in nodes_df.to_dict('records'):
        attr[str(rec.get('stop_id'))] = {
            'stop_name': _txt(rec.get('stop_name')),
            'lat': _num(rec.get('lat')),
            'lon': _num(rec.get('lon')),
            'region': _txt(rec.get('region')),
            'metro': _txt(rec.get('metro')),
        }

    weight_col = next((c for c in ('trip_frequency', 'weight', 'count')
                       if c in edges_df.columns), None)

    D = nx.DiGraph()
    for rec in edges_df.to_dict('records'):
        u, v = str(rec['from_stop']), str(rec['to_stop'])
        D.add_edge(u, v, weight=int(rec[weight_col]) if weight_col else 1)

    # Undirected projection: one edge per unordered pair, weights summed.
    G = nx.Graph()
    for u, v, data in D.edges(data=True):
        if G.has_edge(u, v):
            G[u][v]['weight'] += data['weight']
        else:
            G.add_edge(u, v, weight=data['weight'])

    default = {'stop_name': '', 'lat': None, 'lon': None, 'region': '', 'metro': ''}
    for graph in (G, D):
        for n in graph.nodes():
            graph.nodes[n].update(attr.get(n, default))
    return G, D


G, D = build_graphs(nodes_df, edges_df)
unnamed = sum(1 for n in G.nodes() if not G.nodes[n]['stop_name'])
print(f'undirected G : {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges')
print(f'directed   D : {D.number_of_nodes():,} nodes, {D.number_of_edges():,} edges')
print(f'nodes with no name attribute: {unnamed:,}')

## 6. סטטיסטיקה תיאורית גלובלית

זהו תא המדידה המרכזי. הוא מחשב, במעבר אחד:

* **גודל**: מספרי הצמתים והקשתות עבור הגרף הלא-מכוון והמכוון כאחד.
* **Density** `2m / (n(n-1))`: איזה חלק מכלל זוגות התחנות האפשריים מחובר ישירות. עבור רשת תחבורה מרחבית, מצופה שערך זה יהיה זעיר.
* **סיכום דרגות**: ממוצע, חציון, מינימום ומקסימום של מספר התחנות השכנות.
* **רכיבי קשירות** של `G`, ממוינים לפי גודל, בתוספת חלקן של התחנות השוכנות ברכיב הגדול ביותר - המדד המקובל למידת הפיצול של הרשת עוד לפני שאנו תוקפים אותה.
* **רכיבי קשירות חלשה / חזקה** של `D`. הפער ביניהם מלמד כמה מהרשת מופעלת בשני הכיוונים: רכיב קשירות חזקה גדול משמעו שלרוב זוגות התחנות קיים מסלול חזרה.
* **נקודות חיתוך (articulation points)** ו-**גשרים (bridges)**. צומת חיתוך היא תחנה שהסרתה מגדילה את מספר רכיבי הקשירות; גשר הוא מקטע בעל אותה תכונה. `networkx` מחשבת את שניהם באלגוריתמי DFS בזמן לינארי (רכיבים דו-קשירים לפי Hopcroft-Tarjan, ופירוק לשרשראות עבור גשרים), ולכן תא זה מסתיים בשניות ולא בדקות. אנו עוטפים את `articulation_points` ב-`sorted(set(...))` משום שגרסאות ישנות של `networkx` עלולות היו לפלוט את אותה צומת חיתוך יותר מפעם אחת, דבר שהיה מנפח את הספירה.

התוצאה נכתבת אל `network_summary.json` ואל `tables/network_summary.csv` ומוצגת כטבלה.

In [ ]:
# --- Global descriptive statistics ----------------------------------------
components = sorted(nx.connected_components(G), key=len, reverse=True)
component_sizes = [len(c) for c in components]
largest = component_sizes[0]

# Cut vertices and cut edges: both are linear-time DFS routines.
ap = sorted(set(nx.articulation_points(G)))
br = list(nx.bridges(G))

wcc = list(nx.weakly_connected_components(D))
scc = list(nx.strongly_connected_components(D))

degrees = np.array([d for _, d in G.degree()])

summary = {
    'num_nodes': G.number_of_nodes(),
    'num_edges_undirected': G.number_of_edges(),
    'num_edges_directed': D.number_of_edges(),
    'density': round(nx.density(G), 6),
    'avg_degree': round(float(degrees.mean()), 2),
    'median_degree': float(np.median(degrees)),
    'min_degree': int(degrees.min()),
    'max_degree': int(degrees.max()),
    'num_connected_components': len(components),
    'largest_component_nodes': largest,
    'largest_component_share': round(largest / G.number_of_nodes(), 4),
    'num_weakly_connected': len(wcc),
    'num_strongly_connected': len(scc),
    'largest_scc_nodes': max(len(c) for c in scc),
    'num_articulation_points': len(ap),
    'num_bridges': len(br),
}

with open(STAGE / 'network_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
pd.DataFrame([summary]).to_csv(TABLES / 'network_summary.csv', index=False, encoding='utf-8-sig')

print('saved:', STAGE / 'network_summary.json')
pd.DataFrame({'metric': list(summary.keys()), 'value': list(summary.values())})

## 7. התפלגות הדרגות

דרגתה של תחנה היא מספר התחנות הנבדלות הנגישות ממנה ישירות במקטע אחד. משורטטות שתי תצוגות:

* **הפאנל העליון** - היסטוגרמה לינארית עם ציר y לוגריתמי. הסקאלה הלוגריתמית היא שינוי לטובת הקריאות ביחס לסקריפט המקורי: בציר y לינארי, העמודה של דרגה 2 גבוהה כל כך עד שכל הזנב נעלם.
* **הפאנל התחתון** - אותם נתונים על צירי log-log, המבחן הוויזואלי המקובל להתפלגות בעלת זנב כבד / דמוית power-law. קו ישר כאן עקבי עם התנהגות scale-free. אנו מוסיפים קו רגרסיה בשיטת הריבועים הפחותים (OLS) ומדווחים על שיפועו. זוהי התאמה תיאורית, **ולא** מבחן power-law קפדני (שהיה מחייב אמידת נראות מרבית בתוספת בדיקת טיב התאמה של Kolmogorov-Smirnov); יש להתייחס לשיפוע כאינדיקטור לכובד הזנב בלבד, לא יותר מכך.

ספירות הדרגות נשמרות גם כטבלה, כך שניתן לשחזר את התרשים בלי להריץ מחדש את המחברת.

In [ ]:
# --- Degree distribution: histogram + log-log view -------------------------
deg_counts = pd.Series(degrees).value_counts().sort_index()
deg_counts = deg_counts[deg_counts.index > 0]

deg_table = pd.DataFrame({'degree': deg_counts.index.astype(int),
                          'num_stations': deg_counts.to_numpy().astype(int)})
deg_table['share'] = (deg_table['num_stations'] / len(degrees)).round(5)
deg_table.to_csv(TABLES / 'degree_distribution.csv', index=False, encoding='utf-8-sig')

x = np.log10(deg_table['degree'].to_numpy(dtype=float))
y = np.log10(deg_table['num_stations'].to_numpy(dtype=float))
slope, intercept = np.polyfit(x, y, 1)

fig, axes = plt.subplots(2, 1, figsize=(8, 10))

axes[0].hist(degrees, bins=DEG_HIST_BINS, color='#2563eb', edgecolor='white', linewidth=0.3)
axes[0].set_yscale('log')
axes[0].set_xlabel('Degree (number of neighbouring stations)')
axes[0].set_ylabel('Number of stations (log scale)')
axes[0].set_title('Degree distribution - Israel public transport network')

axes[1].scatter(x, y, s=12, color='#dc2626', alpha=0.65, label='observed')
axes[1].plot(x, slope * x + intercept, color='#111827', linewidth=1.2, linestyle='--',
             label=f'least-squares fit, slope = {slope:.2f}')
axes[1].set_xlabel('log10(degree)')
axes[1].set_ylabel('log10(number of stations)')
axes[1].set_title('Log-log degree distribution (heavy-tail check)')
axes[1].legend()

plt.tight_layout()
plt.savefig(FIGURES / 'degree_distribution.png', dpi=FIG_DPI)
plt.show()

print(f'degree range: {degrees.min()} to {degrees.max()}, mean {degrees.mean():.2f}, median {np.median(degrees):.0f}')
print(f'stations with degree <= 2: {(degrees <= 2).sum():,} ({(degrees <= 2).mean():.1%})')
print(f'log-log least-squares slope: {slope:.2f}')

## 8. רכיבי קשירות

רכיב קשירות הוא קבוצה מקסימלית של תחנות שניתן להגיע מכל אחת מהן אל האחרות דרך רצף כלשהו של מקטעים. אילו הייתה הרשת רכיב יחיד, כל תחנה בארץ הייתה נגישה מכל תחנה אחרת. אך אין זה המצב - קיימים כמה אשכולות מבודדים קטנים (בדרך כלל שירותים מקומיים שאינם נוגעים כלל ברשת הארצית במופע הנתונים הזה). תרשים העמודות מציג את הרכיבים הגדולים ביותר על ציר y לוגריתמי, משום שהפער בין הרכיב הענק לשאר משתרע על פני ארבעה סדרי גודל ואחרת לא היה קריא. הרשימה המלאה של גדלי הרכיבים נשמרת אל `tables/component_sizes.csv`.

In [ ]:
# --- Connected component sizes --------------------------------------------
comp_table = pd.DataFrame({'component_rank': range(1, len(component_sizes) + 1),
                           'num_stations': component_sizes})
comp_table['share'] = (comp_table['num_stations'] / G.number_of_nodes()).round(6)
comp_table.to_csv(TABLES / 'component_sizes.csv', index=False, encoding='utf-8-sig')

top_components = component_sizes[:TOP_N]
bar_colors = ['#2563eb' if i == 0 else '#94a3b8' for i in range(len(top_components))]

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(range(1, len(top_components) + 1), top_components, color=bar_colors)
ax.set_yscale('log')
ax.set_xticks(range(1, len(top_components) + 1))
ax.set_xlabel('Connected component (rank by size)')
ax.set_ylabel('Number of stations (log scale)')
ax.set_title(f'Connected component sizes - top {len(top_components)} of {len(component_sizes)}')
plt.tight_layout()
plt.savefig(FIGURES / 'components_summary_bar.png', dpi=FIG_DPI)
plt.show()

print(f'{len(component_sizes)} components; largest holds '
      f"{largest:,} stations ({largest / G.number_of_nodes():.2%})")
comp_table.head(TOP_N)

## 9. נקודות חיתוך וגשרים

אלו הן נקודות הכשל הבודדות המבניות, והסיבה שמחברת זו חשובה לשאלת החוסן של הפרויקט.

* **נקודת חיתוך (articulation point)**, כלומר צומת חיתוך, היא תחנה שהסרתה מפצלת את הרכיב שלה לשני חלקים או יותר. נוסעים בצד אחד מאבדים כל קישוריות לצד השני, ולא רק מקבלים מסלול איטי יותר.
* **גשר (bridge)**, כלומר קשת חיתוך, הוא מקטע בעל אותה תכונה - החיבור הפיזי היחיד בין שני חלקים של הרשת.

שתי הטבלאות מיוצאות במלואן, מועשרות בשם התחנה, קואורדינטות, מחוז ודרגה (עבור נקודות חיתוך) או בשמות הקצוות ובתדירות הנסיעות (עבור גשרים), כך שמחברות מאוחרות יותר והדוח הכתוב יוכלו להצליב אליהן. מיון הגשרים לפי תדירות הנסיעות חושף מיד את המקטעים חד-החיבור הנושאים את נפח השירות הגדול ביותר - קשתות החיתוך בעלות ההשפעה הגבוהה ביותר.

In [ ]:
# --- Export cut vertices and cut edges -------------------------------------
ap_df = pd.DataFrame([{
    'stop_id': n,
    'stop_name': G.nodes[n]['stop_name'],
    'lat': G.nodes[n]['lat'],
    'lon': G.nodes[n]['lon'],
    'region': G.nodes[n]['region'],
    'metro': G.nodes[n]['metro'],
    'degree': G.degree(n),
} for n in ap])
ap_df = ap_df.sort_values('degree', ascending=False).reset_index(drop=True)
ap_df.to_csv(TABLES / 'articulation_points.csv', index=False, encoding='utf-8-sig')

br_df = pd.DataFrame([{
    'from_stop': u,
    'to_stop': v,
    'from_name': G.nodes[u]['stop_name'],
    'to_name': G.nodes[v]['stop_name'],
    'trip_frequency': int(G[u][v].get('weight', 1)),
} for u, v in br])
br_df = br_df.sort_values('trip_frequency', ascending=False).reset_index(drop=True)
br_df.to_csv(TABLES / 'bridges.csv', index=False, encoding='utf-8-sig')

print(f'articulation points : {len(ap_df):,} '
      f"({len(ap_df) / G.number_of_nodes():.2%} of all stations)")
print(f'bridges             : {len(br_df):,} '
      f"({len(br_df) / G.number_of_edges():.2%} of all undirected edges)")
print('\nHighest-traffic bridges:')
display(br_df.head(TOP_N))
print('Highest-degree articulation points:')
ap_df.head(TOP_N)

## 10. אילו תחנות הן צומתי החיתוך המשמעותיות ביותר?

לא כל נקודות החיתוך שוות בחשיבותן. צומת חיתוך בעלת דרגה 2 מנתקת ענף מבוי סתום של כמה תחנות; צומת חיתוך בעלת דרגה 30 היא צומת מרכזית (hub) שאובדנה היה מבודד תת-רשת שלמה. תרשים העמודות האופקי שלהלן מדרג את נקודות החיתוך המובילות לפי דרגה, תוך שימוש בשמות התחנות העבריים האמיתיים (המוצגים כהלכה הודות ל-patch ה-bidi שהותקן בסעיף 3). זוהי הרשימה המצומצמת שעליה אמורות להתמקד מחברות החוסן המאוחרות יותר.

In [ ]:
# --- Top articulation points by degree -------------------------------------
top_ap = ap_df.head(TOP_N).iloc[::-1]   # reversed so the largest ends up on top
labels = [f'{name} ({sid})' for name, sid in zip(top_ap['stop_name'], top_ap['stop_id'])]

fig, ax = plt.subplots(figsize=(9, 8))
ax.barh(range(len(top_ap)), top_ap['degree'].to_numpy(), color='#7c3aed')
ax.set_yticks(range(len(top_ap)))
ax.set_yticklabels(labels, fontsize=9)
ax.set_xlabel('Degree (number of neighbouring stations)')
ax.set_title(f'Top {len(top_ap)} articulation points by degree')
plt.tight_layout()
plt.savefig(FIGURES / 'top_articulation_points.png', dpi=FIG_DPI)
plt.show()

## 11. סקירה גאוגרפית של הרשת

פיזור (scatter) של כל תחנה לפי קו האורך / קו הרוחב האמיתי שלה, כשהצבע והגודל נקבעים לפי הדרגה, מעניק תחושה מיידית היכן הרשת צפופה (רצועת החוף תל אביב - חיפה וירושלים) והיכן היא מידללת (הנגב). מדובר בפיזור lat/lon פשוט ולא במפה מוטלת (projected), ולכן אנו קובעים את יחס הממדים ל-`1 / cos(mean latitude)` כדי לשמר את צורת המדינה בקירוב נכון במקום מתוחה לרוחב.

סינון הקואורדינטות כאן הוא תיקון מכוון לסקריפט המקורי, שהשתמש ב-`if d.get("lat") and d.get("lon")`. בדיקה זו מוערכת כ-falsy עבור קואורדינטה שערכה בדיוק `0.0`, וגרוע מכך - מוערכת כ-truthy עבור `NaN`, כך שקואורדינטות חסרות היו משורטטות. במקום זאת אנו בודקים מפורשות קיומו של ערך נוכח וסופי.

In [ ]:
# --- Network overview map (lat/lon scatter) --------------------------------
def has_coords(data):
    """True only when both coordinates are present and finite (0.0 included)."""
    lat, lon = data.get('lat'), data.get('lon')
    return lat is not None and lon is not None and np.isfinite(lat) and np.isfinite(lon)

coord_nodes = [(n, d) for n, d in G.nodes(data=True) if has_coords(d)]
if not coord_nodes:
    raise ValueError('No station carries usable coordinates - check nodes.csv from notebook 02.')

lats = [d['lat'] for _, d in coord_nodes]
lons = [d['lon'] for _, d in coord_nodes]
degs = [G.degree(n) for n, _ in coord_nodes]
max_deg = max(degs)
sizes = [2 + 30 * (d / max_deg) for d in degs]

fig, ax = plt.subplots(figsize=(8, 11))
sc = ax.scatter(lons, lats, c=degs, s=sizes, cmap='viridis', alpha=MAP_ALPHA, linewidths=0)
plt.colorbar(sc, ax=ax, label='Degree')
ax.set_aspect(1 / np.cos(np.deg2rad(float(np.mean(lats)))))
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Israel public transport stations\n'
             f'({len(coord_nodes):,} active stops with coordinates)')
plt.tight_layout()
plt.savefig(FIGURES / 'network_overview_map.png', dpi=FIG_DPI)
plt.show()

print(f'stations with coordinates: {len(coord_nodes):,} of {G.number_of_nodes():,}')

## 12. תחנות לפי מחוז

לסיום, ספירה פשוטה של התחנות הפעילות לכל מחוז מנהלי, המתרגמת את הגאוגרפיה שלעיל למספרים ומשמשת בדיקת שפיות שימושית לכך שתכונת המחוז שרדה את פעולת ה-join במחברת 02. צבעי העמודות נוצרים מפלטת seaborn שגודלה נגזר ממספר המחוזות הקיימים בפועל - הסקריפט המקורי קיבע ארבעה צבעים, מה שהיה גורם למחזוריות שקטה או למחסור בצבעים אילו השתנתה קבוצת המחוזות. שמות המחוזות הם בעברית ומוצגים דרך patch ה-bidi.

In [ ]:
# --- Stations per region ---------------------------------------------------
region_counts = (pd.Series([G.nodes[n]['region'] or 'Unknown' for n in G.nodes()])
                 .value_counts())
region_table = region_counts.rename_axis('region').reset_index(name='num_stations')
region_table['share'] = (region_table['num_stations'] / G.number_of_nodes()).round(4)
region_table.to_csv(TABLES / 'stops_by_region.csv', index=False, encoding='utf-8-sig')

palette = sns.color_palette('deep', len(region_counts))
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar([str(r) for r in region_counts.index], region_counts.to_numpy(), color=palette)
ax.set_xlabel('Region')
ax.set_ylabel('Number of stations')
ax.set_title('Active stations by administrative region')
for bar, val in zip(bars, region_counts.to_numpy()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
            f'{val:,}', ha='center', va='bottom', fontsize=10)
ax.margins(y=0.12)
plt.tight_layout()
plt.savefig(FIGURES / 'stops_by_region.png', dpi=FIG_DPI)
plt.show()

region_table

## מסקנות

* **הרשת גדולה ודלילה במיוחד.** כ-30,463 תחנות פעילות ו-51,772 מקטעים לא-מכוונים מניבים density בסביבות 0.0001 ודרגה ממוצעת של כ-3.4 - תחנה טיפוסית נוגעת רק בשלוש או ארבע תחנות אחרות. זהו מצב נורמלי לרשת תחבורה מרחבית: הגאוגרפיה, ולא הפופולריות, היא שמגבילה מי יכול להיות שכן.
* **למעשה מדובר ברשת אחת.** קיימים כ-12 רכיבי קשירות, אך הגדול שבהם מכיל כ-99.3% מכלל התחנות; היתר הם אשכולות מבודדים זעירים. לפיכך כל אמירה על חוסן במחברות הבאות נוגעת לרכיב הענק, ו"הרשת התפצלה" חייב להיות בעל משמעות חזקה יותר מ"כמה תחנות נשרו".
* **התפלגות הדרגות בעלת זנב כבד, אך אין להגזים בטענה.** רוב התחנות ממוקמות בדרגה 2 (תחנה על קו), בעוד מספר קטן של צומתי מעבר מגיעות לעשרות שכנות. תרשים ה-log-log לינארי בקירוב לאורך עיקר התחום, דבר ה*עקבי עם* התנהגות דמוית scale-free - אך שיפוע הריבועים הפחותים שהודפס לעיל הוא התאמה תיאורית, לא מעריך power-law מאומת, והזנב קצר. אנו מדווחים על כך כ"בעלת זנב כבד", לא כ"scale-free".
* **נקודות כשל בודדות אכן קיימות, אך הן מיעוט קטן.** כ-900 נקודות חיתוך (כ-3% מהתחנות) ו-971 גשרים (פחות מ-2% מהקשתות) מהווים נקודות כשל בודדות מבניות. שתי הסתייגויות כנות: רבות מהן הן ענפי מבוי סתום בעלי דרגה נמוכה, שאובדנם מנתק רק קומץ תחנות - ומשום כך סעיף 10 מדרג אותן לפי דרגה; וזוהי תפיסת כשל טופולוגית גרידא - היא מתעלמת ממספר הנוסעים המשתמשים בפועל במקטע, וזה בדיוק מה שמחברות ה-centrality והחוסן מוסיפות בהמשך.
* **רשת דו-כיוונית ברובה.** לגרף המכוון יש רק מעט יותר קשתות מאשר להיטל הלא-מכוון, ורכיב הקשירות החזקה הגדול ביותר מכסה את רוב הרשת, כלומר עבור הרוב המכריע של זוגות התחנות קיים מסלול חזרה. מקטעים המופעלים בכיוון אחד בלבד הם היוצא מן הכלל.
* **הגאוגרפיה שולטת.** התחנות מרוכזות בעיקר במחוז המרכז וברצועה המטרופולינית של החוף; הפריפריה הדרומית דלילה. זהו הרקע המבני שעל פיו יש לקרוא את הממצאים החברתיים-כלכליים במחברות הבאות.